
# Projeto Eixo 7: Machine Learning e Cybersecurity - Pré-processamento e Análise Exploratória de Dados do dataset de Teste

## A) Importação das Bibliotecas
---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.manifold import TSNE

## B) Criação do DataFrame
---

O DataFrame, `df` é criado usando a função de `Pandas` específica para arquivos `.parquet`, chamada `pd.read_csv()`, uma vez que o dataset `UNSW_NB15_training-set.parquet` está neste formato.

In [4]:
df= pd.read_parquet('../dataset/original/UNSW_NB15_testing-set.parquet')
df.head()

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.000011,udp,-,INT,2,0,496,0,90909.09375,180363632.0,...,0,0,1,1,0,0,0,0,Normal,0
1,0.000008,udp,-,INT,2,0,1762,0,125000.00000,881000000.0,...,0,0,1,1,0,0,0,0,Normal,0
2,0.000005,udp,-,INT,2,0,1068,0,200000.00000,854400000.0,...,0,0,1,1,0,0,0,0,Normal,0
3,0.000006,udp,-,INT,2,0,900,0,166666.65625,600000000.0,...,0,0,2,1,0,0,0,0,Normal,0
4,0.000010,udp,-,INT,2,0,2126,0,100000.00000,850400000.0,...,0,0,2,1,0,0,0,0,Normal,0


## C) Manipulação e Pré-processamento de Dados
---

### 1. Conferência dos atributos
---

O método `.info()` mostra todos os atributos (colunas) do DataFrame, junto à quantidade de valores não nulos e tipagem de cada um, além de gasto de memória do df.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 82332 entries, 0 to 82331
Data columns (total 36 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   dur                82332 non-null  float32 
 1   proto              82332 non-null  category
 2   service            82332 non-null  category
 3   state              82332 non-null  category
 4   spkts              82332 non-null  int16   
 5   dpkts              82332 non-null  int16   
 6   sbytes             82332 non-null  int32   
 7   dbytes             82332 non-null  int32   
 8   rate               82332 non-null  float32 
 9   sload              82332 non-null  float32 
 10  dload              82332 non-null  float32 
 11  sloss              82332 non-null  int16   
 12  dloss              82332 non-null  int16   
 13  sinpkt             82332 non-null  float32 
 14  dinpkt             82332 non-null  float32 
 15  sjit               82332 non-null  float32 
 16  djit           

### 2. Renomeando os atributos
---

A adequação dos nomes dos atributos é feita com o atributo `.rename()`. Para tal, há a necessidade de explicitar o parâmetro `columns` e passar como argumento os nomes antigos e novos dos atributos, em formato de `dicionário` ou `JSON`.

É feita para uma maior comodidade e compreensão no contexto de `Manipulação de Dados` e `Análise Exploratória de Dados`.

In [6]:
df = df.rename(columns={
    'dur':'duracao_conexao_seg',
    'proto':'protocolo_rede',
    'service':'servico_conexao',
    'state':'estado_conexao',
    'spkts':'num_pacotes_origem',
    'dpkts':'num_pacotes_destino',
    'sbytes':'total_bytes_origem',
    'dbytes':'total_bytes_destino',
    'rate':'taxa_pacote',
    'sload':'carga_origem',
    'dload':'carga_destino',
    'sloss':'pacotes_perdidos_origem',
    'dloss':'pacotes_perdidos_destino',
    'sinpkt':'tempo_medio_pacote_origem',
    'dinpkt':'bytes_por_pacote_destino',
    'sjit':'var_lat_pacotes_origem',
    'djit':'var_lat_pacotes_destino',
    'swin':'tamanho_janela_tcp_origem',
    'stcpb':'num_sequencia_tcp_origem',
    'dtcpb':'num_sequencia_tcp_destino',
    'dwin':'tamanho_janela_tcp_destino',
    'tcprtt':'rtt_handshake_tcp',
    'synack':'tempo_syn_synack',
    'ackdat':'tempo_synack_ack',
    'smean':'tamanho_medio_pacotes_origem',
    'dmean':'tamanho_medio_pacotes_destino',
    'trans_depth':'profundidade_transacao_http',
    'response_body_len':'tamanho_corpo_resposta_http',
    'ct_src_dport_ltm':"conexoes_recentes_ip_porta_destino",
    'ct_dst_sport_ltm':"conexoes_recentes_ip_porta_origem",
    'is_ftp_login':'ftp_autenticada',
    'ct_ftp_cmd':'num_comandos_ftp',
    'ct_flw_http_mthd':'num_metodos_http',
    'is_sm_ips_ports':'ip_porta_origem_destino_iguais',
    'attack_cat':'categoria_ataque',
    'label':'label'
})

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 82332 entries, 0 to 82331
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype   
---  ------                              --------------  -----   
 0   duracao_conexao_seg                 82332 non-null  float32 
 1   protocolo_rede                      82332 non-null  category
 2   servico_conexao                     82332 non-null  category
 3   estado_conexao                      82332 non-null  category
 4   num_pacotes_origem                  82332 non-null  int16   
 5   num_pacotes_destino                 82332 non-null  int16   
 6   total_bytes_origem                  82332 non-null  int32   
 7   total_bytes_destino                 82332 non-null  int32   
 8   taxa_pacote                         82332 non-null  float32 
 9   carga_origem                        82332 non-null  float32 
 10  carga_destino                       82332 non-null  float32 
 11  pacotes_perdidos_origem             823

### 3. Conferência de Valores e Nulos
---

In [8]:
for col in df.columns:
    print(f'\nValores do atributo {col}:\n')
    print(df[col].value_counts())
    print('\n')
    print('='*40)


Valores do atributo duracao_conexao_seg:

duracao_conexao_seg
0.000009    5546
0.000008    4938
0.000003    4111
0.000004    3336
0.000011    3112
            ... 
0.524434       1
0.947039       1
0.811914       1
5.167410       1
1.106101       1
Name: count, Length: 39888, dtype: int64



Valores do atributo protocolo_rede:

protocolo_rede
tcp     43095
udp     29418
unas     3515
arp       987
ospf      676
        ...  
wsn        32
xtp        32
zero       32
ib         31
igmp       30
Name: count, Length: 131, dtype: int64



Valores do atributo servico_conexao:

servico_conexao
-           47153
dns         21367
http         8287
smtp         1851
ftp          1552
ftp-data     1396
pop3          423
ssh           204
ssl            30
snmp           29
dhcp           26
radius          9
irc             5
Name: count, dtype: int64



Valores do atributo estado_conexao:

estado_conexao
FIN    39339
INT    34163
CON     6982
REQ     1842
ACC        4
CLO        1
RST        

Valores nulos geram erro de processamento, por isso devem ser tratados.

No dataset escolhido, nenhuma coluna apresentou tais valores.

In [9]:
df.isna().sum()

duracao_conexao_seg                   0
protocolo_rede                        0
servico_conexao                       0
estado_conexao                        0
num_pacotes_origem                    0
num_pacotes_destino                   0
total_bytes_origem                    0
total_bytes_destino                   0
taxa_pacote                           0
carga_origem                          0
carga_destino                         0
pacotes_perdidos_origem               0
pacotes_perdidos_destino              0
tempo_medio_pacote_origem             0
bytes_por_pacote_destino              0
var_lat_pacotes_origem                0
var_lat_pacotes_destino               0
tamanho_janela_tcp_origem             0
num_sequencia_tcp_origem              0
num_sequencia_tcp_destino             0
tamanho_janela_tcp_destino            0
rtt_handshake_tcp                     0
tempo_syn_synack                      0
tempo_synack_ack                      0
tamanho_medio_pacotes_origem          0


### 4. Conferência de registros duplicados em todo o DataFrame
---

Manter registros completos duplicados na base de dados que será usada para testes do modelo preditivo pode levar a `overfitting`, onde o modelo acaba "decorando" padrões específicos, que pode levar a um enviezamento de respostas. Portanto, duplicatas devem ser excluídas.

No dataset, há a presença de `32361` registros duplicados.

In [10]:
df.duplicated().sum()

np.int64(32361)

### 5. Conferência de valores únicos por atributo
---

O dataset possui 11 atributos que, por terem poucos valores únicos poderiam ser divididos em `categorias`:
| Atributo | Número de valores únicos |
| :---: | :---: |
| `servico_conexao` | 13 |
| `estado_conexao` | 9 |
| `tamanho_janela_tcp_origem` | 13 |
| `tamanho_janela_tcp_destino` | 7 |
| `profundidade_transacao_http` | 11 |
| `ftp_autenticada` | 4 |
| `num_comandos_ftp` | 4 |
| `num_metodos_http` | 11 |
| `ip_porta_origem_destino_iguais` | 2 |
| `categoria_ataque` | 10 |
| `label` | 2 |

Os demais atributos possuem diversos valores únicos, não podendo ser divididos e, categorias de maneira mais simples.

In [11]:
df.nunique()

duracao_conexao_seg                   39888
protocolo_rede                          131
servico_conexao                          13
estado_conexao                            7
num_pacotes_origem                      420
num_pacotes_destino                     436
total_bytes_origem                     4489
total_bytes_destino                    4034
taxa_pacote                           40557
carga_origem                          42873
carga_destino                         40614
pacotes_perdidos_origem                 253
pacotes_perdidos_destino                311
tempo_medio_pacote_origem             39949
bytes_por_pacote_destino              37596
var_lat_pacotes_origem                39911
var_lat_pacotes_destino               38355
tamanho_janela_tcp_origem                11
num_sequencia_tcp_origem              39219
num_sequencia_tcp_destino             39108
tamanho_janela_tcp_destino               14
rtt_handshake_tcp                     26130
tempo_syn_synack                

### 6. Exclusão de registros duplicados
---

Manter colunas e dados desnecessários é perigoso, pois não trazem nenhum tipo de informação relevante ou características que agregam valor ao modelo preditivo, mas podem ocasionar `enviesamento` e `ruídos` que podem confundí-lo e reduzir sua eficiência e confiabilidade.

O dataset possui `82332 ` registros no total, mas destes, `32361` são duplicados e devem, portanto, ser apagados. 

In [12]:
df.duplicated().sum()

np.int64(32361)

In [13]:
df = df.drop_duplicates()

In [14]:
print(df.shape)

(49971, 36)


### 7. Adequando a tipagem dos dados relativos aos atributos
---

É necessária a adequação da tipagem dos dados relativos aos atributos, para que todo o processo de `Análise Exploratória de Dados` e `Treino e Teste dos Modelos de Aprendizado Supervisionado` aconteçam da melhor maneira.

Tanto a `Matriz de Correlação de Pearson` e a biblioteca `Scikit-Learn` operam somente com valores numéricos. Em caso de haver dados das classes `str`, `object`, `datetime` e `category`, a matriz pode desconsiderá-los e o sklearn pode retornar erro.

Há a necessidade de tratar os atributos categóricos que tenham a tipagem `str` ou `object`. São muito relevantes tanto para as análises descritivas, quanto para as preditivas, por agruparem dados que possam ter grande correlação com o diagnóstico de ataque cibernético. Contudo, para serem usados, devem ser transformados em números.   

A tecnologia escolhida foi o `OrdinalEncoder` de `Scikit Learn` para transformar os dados dos atributos categóricos em números (`float64`) possibilitando seu uso na `Ánalise Exploratória dos Dados` e `Treino e Teste do Modelo de Aprendizado Supervisionado`. Há a vantagem de não aumentar a dimensionalidade do `df` (aumento do número de colunas transformar cada categoria em um atributo novo).

In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 49971 entries, 0 to 82328
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype   
---  ------                              --------------  -----   
 0   duracao_conexao_seg                 49971 non-null  float32 
 1   protocolo_rede                      49971 non-null  category
 2   servico_conexao                     49971 non-null  category
 3   estado_conexao                      49971 non-null  category
 4   num_pacotes_origem                  49971 non-null  int16   
 5   num_pacotes_destino                 49971 non-null  int16   
 6   total_bytes_origem                  49971 non-null  int32   
 7   total_bytes_destino                 49971 non-null  int32   
 8   taxa_pacote                         49971 non-null  float32 
 9   carga_origem                        49971 non-null  float32 
 10  carga_destino                       49971 non-null  float32 
 11  pacotes_perdidos_origem             49971 no

In [16]:
atr_categoricos = ['protocolo_rede', 'servico_conexao', 'estado_conexao', 'categoria_ataque']

In [17]:
encoder = OrdinalEncoder()
df[atr_categoricos] = encoder.fit_transform(df[atr_categoricos])

Aumentar todos os valores `int` e `float` entre 16 e 32 para 64, para garantir que os números não apresentem problemas, já que podem ser bem extensos

- Aumento de `int`:

In [18]:
int_aumentar = ['num_pacotes_origem', 'num_pacotes_destino', 'total_bytes_origem', 'total_bytes_destino', 'pacotes_perdidos_origem', 'pacotes_perdidos_destino','num_sequencia_tcp_origem', 'tamanho_janela_tcp_origem', 'tamanho_janela_tcp_destino',
                'num_sequencia_tcp_destino', 'tamanho_medio_pacotes_origem', 'tamanho_medio_pacotes_destino', 'profundidade_transacao_http', 'tamanho_corpo_resposta_http',]

In [19]:
for x in int_aumentar:
    df[x] = df[x].astype(int)
    print(f'O atributo {x} teve o tipo convertido para int64')

O atributo num_pacotes_origem teve o tipo convertido para int64
O atributo num_pacotes_destino teve o tipo convertido para int64
O atributo total_bytes_origem teve o tipo convertido para int64
O atributo total_bytes_destino teve o tipo convertido para int64
O atributo pacotes_perdidos_origem teve o tipo convertido para int64
O atributo pacotes_perdidos_destino teve o tipo convertido para int64
O atributo num_sequencia_tcp_origem teve o tipo convertido para int64
O atributo tamanho_janela_tcp_origem teve o tipo convertido para int64
O atributo tamanho_janela_tcp_destino teve o tipo convertido para int64
O atributo num_sequencia_tcp_destino teve o tipo convertido para int64
O atributo tamanho_medio_pacotes_origem teve o tipo convertido para int64
O atributo tamanho_medio_pacotes_destino teve o tipo convertido para int64
O atributo profundidade_transacao_http teve o tipo convertido para int64
O atributo tamanho_corpo_resposta_http teve o tipo convertido para int64


- Aumento de `float`:

In [20]:
float_aumentar = ['protocolo_rede', 'servico_conexao', 'estado_conexao', 'categoria_ataque', 'duracao_conexao_seg', 'taxa_pacote', 'carga_origem', 'carga_destino', 'tempo_medio_pacote_origem', 'bytes_por_pacote_destino', 'var_lat_pacotes_origem',
                  'var_lat_pacotes_destino', 'rtt_handshake_tcp', 'tempo_syn_synack', 'tempo_synack_ack', 'tamanho_corpo_resposta_http']

In [21]:
for x in float_aumentar:
  df[x] = df[x].astype(float)
  print(f'O atributo {x} teve o tipo convertido para float64')

O atributo protocolo_rede teve o tipo convertido para float64
O atributo servico_conexao teve o tipo convertido para float64
O atributo estado_conexao teve o tipo convertido para float64
O atributo categoria_ataque teve o tipo convertido para float64
O atributo duracao_conexao_seg teve o tipo convertido para float64
O atributo taxa_pacote teve o tipo convertido para float64
O atributo carga_origem teve o tipo convertido para float64
O atributo carga_destino teve o tipo convertido para float64
O atributo tempo_medio_pacote_origem teve o tipo convertido para float64
O atributo bytes_por_pacote_destino teve o tipo convertido para float64
O atributo var_lat_pacotes_origem teve o tipo convertido para float64
O atributo var_lat_pacotes_destino teve o tipo convertido para float64
O atributo rtt_handshake_tcp teve o tipo convertido para float64
O atributo tempo_syn_synack teve o tipo convertido para float64
O atributo tempo_synack_ack teve o tipo convertido para float64
O atributo tamanho_corp

In [22]:
df.info()

<class 'pandas.DataFrame'>
Index: 49971 entries, 0 to 82328
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   duracao_conexao_seg                 49971 non-null  float64
 1   protocolo_rede                      49971 non-null  float64
 2   servico_conexao                     49971 non-null  float64
 3   estado_conexao                      49971 non-null  float64
 4   num_pacotes_origem                  49971 non-null  int64  
 5   num_pacotes_destino                 49971 non-null  int64  
 6   total_bytes_origem                  49971 non-null  int64  
 7   total_bytes_destino                 49971 non-null  int64  
 8   taxa_pacote                         49971 non-null  float64
 9   carga_origem                        49971 non-null  float64
 10  carga_destino                       49971 non-null  float64
 11  pacotes_perdidos_origem             49971 non-null  int64

### 8. Normalização de Features
---

As análises dos atributos, indicaram variações de valores muito bruscas entre eles. Para isso, opta-se por realizar a `Normalização das Features`. O método de escalonamento escolhido foi o `Robust Scaler`, por ser resistente aos `Outliers`, que foram preservados, presentes na maioria dos atributos e a `Distribuição Não Normal`, presentes em todos.

In [113]:
escalonador = RobustScaler()

# Lista que contém os atributos categóricos e/ou com poucos valores únicos
atr_nao_categoricos = df.drop(columns=['protocolo_rede','tamanho_janela_tcp_origem', 'servico_conexao','estado_conexao','categoria_ataque'])

for col in atr_nao_categoricos:
    _, valor_min, valor_max, _, _, faixa = pm.gerar_dispersao(df, col)
    print(f'\n\nAtributo {col.upper()}:\n')
    print(f'O atributo {col} possui amplitude entre {valor_min} e {valor_max} e magnitude {faixa} antes da Normalização com RobustScaler.\n')
    
    normalizacao = pp.normalizar_feature_rs(df, col, escalonador, div_max=True)
    df[col] = normalizacao
    
    _, valor_min, valor_max, _, _, faixa = pm.gerar_dispersao(df, col)
    print(f'O atributo {col} foi normalizado. Sua amplitude atual está entre {valor_min} e {valor_max} e sua magnitude, em {faixa}.\n')



Atributo DURACAO_CONEXAO_SEG:

O atributo duracao_conexao_seg possui amplitude entre 0.0 e 59.9999885559082 e magnitude 59.9999885559082 antes da Normalização com RobustScaler.

O atributo duracao_conexao_seg foi normalizado. Sua amplitude atual está entre -0.008019671521446376 e 1.0 e sua magnitude, em 1.0080196715214464.



Atributo NUM_PACOTES_ORIGEM:

O atributo num_pacotes_origem possui amplitude entre 1 e 10646 e magnitude 10645 antes da Normalização com RobustScaler.

O atributo num_pacotes_origem foi normalizado. Sua amplitude atual está entre -0.0008461827754795035 e 1.0 e sua magnitude, em 1.0008461827754795.



Atributo NUM_PACOTES_DESTINO:

O atributo num_pacotes_destino possui amplitude entre 0 e 11018 e magnitude 11018 antes da Normalização com RobustScaler.

O atributo num_pacotes_destino foi normalizado. Sua amplitude atual está entre -0.0007266121707538601 e 1.0 e sua magnitude, em 1.0007266121707539.



Atributo TOTAL_BYTES_ORIGEM:

O atributo total_bytes_origem pos

## D) Criação do Dataset Pré-processado de Teste
---

Este dataset será usado em conjunto com o de treino nos modelos preditivos.

In [117]:
df.to_parquet('../dataset/pre_processado/teste_pre_processado.parquet')